<a href="https://colab.research.google.com/github/Julianmoralez/Retos-Colab/blob/Retos-Colab/AnalisisLavadoActivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from ast import Import

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns


from sklearn.naive_bayes import GaussianNB

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score

from sklearn.metrics import confusion_matrix

from sklearn.metrics import classification_report

Se cargan los datos de trabajo de la base de datos

In [ ]:
from os import read
nxl='/content/drive/MyDrive/Analitica de negocios/Copia de 2. LavadoActivos.xlsx'
XDB=pd.read_excel(nxl,sheet_name=0)
XDB.head()
XDB.info()
XDB.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3150 entries, 0 to 3149
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Unnamed: 0                         3150 non-null   int64  
 1   edad                               3150 non-null   int64  
 2   ingresos_usd                       3150 non-null   float64
 3   gastos_usd                         3150 non-null   float64
 4   num_tarjetas_credito               3150 non-null   int64  
 5   monto_transado_tarjetas_usd        3150 non-null   float64
 6   porcentaje_crecimiento_patrimonio  3150 non-null   float64
 7   lavado_activos                     3150 non-null   int64  
dtypes: float64(4), int64(4)
memory usage: 197.0 KB


,Unnamed: 0,edad,ingresos_usd,gastos_usd,num_tarjetas_credito,monto_transado_tarjetas_usd,porcentaje_crecimiento_patrimonio,lavado_activos
count,3150.000,3150.000000,3.150000e+03,3.150000e+03,3150.000000,3150.000000,3150.000000,3150.000000
mean,1574.500,45.125397,2.539894e+04,1.393281e+04,1.972698,12138.324124,44.726495,0.294286
std,909.471,14.666110,6.821424e+04,3.926671e+04,1.345358,36996.070126,14.411384,0.455793
min,0.000,20.000000,6.286000e+01,3.805000e+01,0.000000,46.570000,20.010000,0.000000
25%,787.250,32.250000,3.084438e+03,1.620003e+03,1.000000,1507.602500,32.320000,0.000000
50%,1574.500,45.000000,8.438090e+03,4.517480e+03,2.000000,3857.720000,44.410000,0.000000
75%,2361.750,57.000000,2.255623e+04,1.204981e+04,3.000000,10544.045000,56.800000,1.000000
max,3149.000,70.000000,1.956275e+06,1.025902e+06,5.000000,986952.960000,69.970000,1.000000


In [ ]:
# Variable objetivo (lo que queremos predecir)
y = XDB['lavado_activos']

# Variables predictoras (todas menos la objetivo, incluyendo 'Unnamed: 0' si existe)
x = XDB.drop(['lavado_activos', 'Unnamed: 0'], axis=1, errors='ignore')
x.head()
y.head()

,lavado_activos
0,1
1,1
2,0
3,0
4,0


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x, # Use the corrected 'x' here
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

(2205, 6)
(945, 6)


In [ ]:
from sklearn.naive_bayes import GaussianNB

mnb = GaussianNB()
mnb.fit(X_train, y_train)


GaussianNB()

In [ ]:
y_pred = mnb.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[655  12]
 [ 47 231]]


In [ ]:
VN = cm[0,0]
FP = cm[0,1]
FN = cm[1,0]
VP = cm[1,1]

TDatos = len(X_test)

Exactitud = (VP + VN) / TDatos
TasaError = (FP + FN) / TDatos
Sensibilidad = VP / (VP + FN)
Especificidad = VN / (VN + FP)
Precision = VP / (VP + FP)
PrediccionNegativa = VN / (VN + FN)
print(cm)
print("Exactitud:", Exactitud)
print("Tasa Error:", TasaError)
print("Sensibilidad (Recall):", Sensibilidad)
print("Especificidad:", Especificidad)
print("Precision:", Precision)
print("Prediccion Negativa:", PrediccionNegativa)

[[655  12]
 [ 47 231]]
Exactitud: 0.9375661375661376
Tasa Error: 0.06243386243386243
Sensibilidad (Recall): 0.8309352517985612
Especificidad: 0.9820089955022488
Precision: 0.9506172839506173
Prediccion Negativa: 0.9330484330484331


Se logró una exactitud del 93.75% y una sensibilidad del 83%, demostrando su capacidad para identificar clientes con alto riesgo de lavado de activos. No obstante, se identificó un número reducido de falsos negativos, por lo que se recomienda su uso como herramienta de apoyo a la toma de decisiones del área de cumplimiento, complementando los controles tradicionales.

Nueva del modelo con posible culpable

In [ ]:
Prueba_cliente = pd.DataFrame([{
    'edad': 42,
    'ingresos_usd': 9500,
    'gastos_usd': 4800,
    'num_tarjetas_credito': 2,
    'monto_transado_tarjetas_usd': 12000,
    'porcentaje_crecimiento_patrimonio': 68
}])

In [ ]:
prediccion= mnb.predict(Prueba_cliente)
prediccion #Significaria que no hay riesgo de que con estas condiciones el cliente sea peligroso

array([0])

In [ ]:
probabilidad = mnb.predict_proba(Prueba_cliente)
probabilidad


array([[0.97167849, 0.02832151]])

Significando que la probabilidad de que el cliente no sea riesgoso es del 97,1%,y el modelo clasifica al cliente como no riesgoso

In [ ]:
Prueba_cliente2= pd.DataFrame([{
    'edad':56,
    'ingresos_usd':15000,
    'gastos_usd':50000,
    'num_tarjetas_credito':3,
    'monto_transado_tarjetas_usd':2000
    ,'porcentaje_crecimiento_patrimonio':160
}])

Prediccion2=mnb.predict(Prueba_cliente2)
Prediccion2

Probabilidad2=mnb.predict_proba(Prueba_cliente2)
Probabilidad2
#Significa que esta casi segurisimo de que esta persona LavaActivos


array([[1.17732894e-42, 1.00000000e+00]])